# CamemBERT

In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    f1_score, roc_auc_score, average_precision_score,
    classification_report
)
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import re
import codecs
from sklearn.model_selection import train_test_split
from gensim.models import FastText, KeyedVectors
from nltk.tag.crf import CRFTagger
import pycrfsuite
import numpy as np
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report, accuracy_score
import warnings
warnings.filterwarnings("ignore")
from gensim.models import Word2Vec
from scipy.sparse import hstack, csr_matrix
from transformers import CamembertForSequenceClassification, CamembertTokenizer
import torch
from torch.utils.data import Dataset, DataLoader
from camembert_finetune import *

## Load Text
alllabs: 
- 1: Chirac
- -1: Mitterrand

In [2]:
def load_pres(fname):
    alltxts = []
    alllabs = []
    s=codecs.open(fname, 'r','utf-8') # pour régler le codage
    while True:
        txt = s.readline()
        if(len(txt))<5:
            break
        #
        lab = re.sub(r"<[0-9]*:[0-9]*:(.)>.*","\\1",txt)
        txt = re.sub(r"<[0-9]*:[0-9]*:.>(.*)","\\1",txt)
        if lab.count('M') > 0:
            alllabs.append(1)   # Mitterrand = 1
        else:
            alllabs.append(0)   # Chirac = 0
        alltxts.append(txt)
    return alltxts,alllabs

In [3]:
fname = "../../data/corpus.tache1.learn.utf8"
alltxts, alllabs = load_pres(fname)

len(alltxts), len(alllabs)

(57413, 57413)

In [4]:
X_train, X_val, y_train, y_val = train_test_split(
    alltxts, alllabs,
    test_size=0.2,
    random_state=42,
    stratify=alllabs      # preserve 87/13 ratio in both splits
)

print(f"Train: {len(X_train)} sentences | Val: {len(X_val)} sentences")
print(f"Train Mitterrand: {sum(y_train)} ({100*sum(y_train)/len(y_train):.1f}%)")

Train: 45930 sentences | Val: 11483 sentences
Train Mitterrand: 6018 (13.1%)


In [5]:
trainer, tokenizer, model = train_camembert(X_train, y_train, X_val, y_val)
# Uncomment when test set is available:
# test_texts = [...]
# generate_submission(trainer, tokenizer, test_texts, "submission_camembert.txt")

Loading CamemBERT...


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1492.40it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
CamembertForSequenceClassification LOAD REPORT from: camembert-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consid

Detected encoder attribute: 'roberta'
Strategy: 'top_layers'
Trainable params: 14,767,874 / 110,623,490 (13.3%)
Train distribution → Chirac: 39912 | Mitterrand: 6018
Class weights → Chirac: 0.575 | Mitterrand: 3.816

Training...


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
val_probs = evaluate_camembert(trainer, tokenizer, X_val, y_val)